In [ ]:
import numpy as np;
from tensorflow.keras.datasets import mnist;
import random as rnd;
import sys;
import math;

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum

    return normalised.astype(np.half)

train_images_vectored = np.array([normalize_and_vectorize_image(img) for img in train_images])
test_images_vectored = np.array([normalize_and_vectorize_image(img) for img in test_images])

def bin_image(image, threshold = 122):
    """
    Turn an image into it's binned version with unit8's of values 1 and 0. threshold determines below what values a pixel is condicered empty (0)
    """
    size = len(image)
    binned_image = np.zeros((size,size), dtype="uint8")
    for y_i, y_val in enumerate(image):
        for x_i, x_val in enumerate(y_val):
            if x_val > threshold:
                binned_image[y_i, x_i] = 1
            else:
                binned_image[y_i, x_i] = 0
    return binned_image

def binary_encode_np(image):
    image = bin_image(image)
    return np.packbits(image)

def binary_decode_np(encoded_image):
    unpacked = np.unpackbits(encoded_image)
    size = int(math.sqrt(len(unpacked)))
    return unpacked[: size**2].reshape((size, size))

train_images_binned = [(binary_decode_np(binary_encode_np(im))) for im in train_images]
test_images_binned = [(binary_decode_np(binary_encode_np(im))) for im in test_images]

def order_by_number(dataset_images, dataset_labels):
    """
    Order dataset by labels to that every unique label has a list of values that share the label.
    """
    ordered_images = {0: [], 1: [], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[], 9:[]}
    for i, _ in enumerate(dataset_images):
        if (len(ordered_images[dataset_labels[i]]) > 90): # addition since previous version. a limitor for the amount of images.
            continue
        ordered_images[dataset_labels[i]].append(dataset_images[i])
    return ordered_images

ordered_images = order_by_number(train_images_binned, train_labels)

# Onderzoek MNIST nummer herkenning

## Inhoud
- 1 inleiding
- 2 data begrijpen
- 3 encoding
- 4 feature engineering
- 5 model/algoritme keuze
- 6 eindkeuze
- 7 relectie
- 8 taakverdeling

## 1. Inleiding
In het vak Datastructures and alogirthms zijn in de afgelopen weken verschillende voorspellings algoritmen behandeld. Deze zijn ingezet om getallen uit de mnist dataset te herkennen. In deze slotopdracht wordt een overzicht gemaakt. Hierbij staat de vraag centraal:\
__Welk algoritme is het best geschikt voor cijferherkenning, runnend op de mystery device?__\
De mystery device heeft de volgende specs
- 128 kb ram
- 100 mb opslag

## 2. Data begrijpen

__Wat MNIST is__\
MNIST is een gelabelde dataset van 70000 afbeeldingen van handgeschreven cijfers.
De Mnist dataset in ontwikkeld in de jaren 90 als 'objective' maatstaf voor het bepalen van naukeurigheid van cijferherkenningsprogramma's voor de Amerikaanse overheid. Deze noodzaak was ontstaan door lastig te vergelijken claims van verschillende aanbieders van dit soort software. Vaak met als doel automatisch overheidsformulieren in te lezen.\

__MNIST in cijfers__\
afbeeldingen: 70000 stuks\
afmetingen afbeelding: 28*28 pixels\
verdeling cijfers:



In [ ]:
for k, v in ordered_images.items:
    print(f"{k} amount: {len(v)}")

cijfers zijn in greyscale iedere pixel heeft een waarde van 0-255.

__lastige cijfers__\
sommige cijfers worden gemakkelijker herkent dan anderen.
In voorgaande opdrachten is gebleken dat de 4 en 9 vaak verward worden.

Het valt op dat cijfers die vaak hetzelfde lijken gemakkelijker worden herkent. de 0 en 1 scoren bijvoorbeeld vaak hoog. Verder valt op dat de afbeeldingen zo zijn gemaakt dat het cijfer altijd goed in het midden past. Ondanks dat de MNIST dataset uit 70000 afbeeldingen bestaat zijn sommige pixels nooit gebruikt.

Welke informatie belangrijk is voor het herkennen van een cijfer is lastig te zeggen. Dit is namelijk sterk afhankelijk van de methode waarmee een afbeelding wordt herkent. In de hiervoor gemaakt oprachten blijkt echter dat de beste manier om een afbeelding te voorspellen het gebruiken van de afbeelding zelf is. Over het algemeen blijken geagregeerde gegevens weinig toegeveogde waarde te hebben in de alogoritmen met de hoogste accuracy. Daarnaast heeft een pixelwaarde range van 0-255 weinig effect op de voorspelde waarden.

__Design keuzes__\
Op grond van deze gegevens zijn de volgende keuzes gemaakt.
1. Voor een goede herkenning van cijfers die zelf getekend zijn (en dus niet voldoen aan de MNIST preprocessing) is het van belang zelf een extra preprocessor toe te voegen op de mystery device die de gegeven afbeelding schaalt zodat deze beter binnen de bestaande datapatronen past.
2. Het valt op dat kleurverloop in de afbeeldingen vaak van beperkte toegevoegde waarde is, Zie opdracht van week 3, zie hoofstuk 'Hoeveel afbeeldingen passen op de mystery device?'. Daarom wordt binning ingezet om afbeeldingrote te reduceren in de alogritmen waar dit zinvol is.
3. Om nieuwe afbeeldingen te gebruiken zal op de mystery device een inlees systeem moeten worden ontwikkeld.
Welk algoritme zal worden gebruikt wordt in volgende hoofstukken behandeld.





## 3. Encoding
Welke encoding wordt gebuikt is afhankelijk van welk alogritme wordt gebruikt. Het doel is hierbij telkens een encoding te kiezen waarmee zo hoog mogelijke accuary kan worden bereikt.


### Prestaties van verschillende compressie algoritmen
Gegevens komen uit P opdracht van week 4


In [ ]:
# Verschillende compression methods, gemaakt in week 4.
def binary_encode(image):
    bin_image = bytearray()

    binary_secion: str = ""
    for row in image: # write pixel data one by one
        for pixel in row:
            if pixel > 50:
                binary_secion += "1"
            else:
                binary_secion += "0"

            if len(binary_secion)  == 8:
                encoded_section = int(binary_secion, 2)
                bin_image.append(encoded_section)
                binary_secion = ""

    return bytes(bin_image)

def binary_decode(binary: bytes, size = 28):
    binaries = ""
    for byte in binary:
        binary_section = bin(byte)[2:].zfill(8)
        binaries += binary_section
    one_d_image = []
    for px in binaries:
        one_d_image.append(int(px) * 255)
    image = np.reshape(one_d_image, (size, size))

    return image

def bin_image(image, threshold = 122):
    """
    Turn an image into it's binned version with unit8's of values 1 and 0. threshold determines below what values a pixel is condicered empty (0)
    """
    size = len(image)
    binned_image = np.zeros((size,size), dtype="uint8")
    for y_i, y_val in enumerate(image):
        for x_i, x_val in enumerate(y_val):
            if x_val > threshold:
                binned_image[y_i, x_i] = 1
            else:
                binned_image[y_i, x_i] = 0
    return binned_image


def binary_encode_np(image):
    image = bin_image(image)
    return np.packbits(image)

def binary_decode_np(encoded_image):
    unpacked = np.unpackbits(encoded_image)
    size = int(math.sqrt(len(unpacked)))
    return unpacked[: size**2].reshape((size, size))

def compres_image(image, new_size = 14):
    """
    Reduce the resolution of an image
    """
    originial_size = len(image)
    step = originial_size / new_size
    steps = [s * step for s in range(new_size)]
    new_image = np.zeros((new_size, new_size), dtype="uint8")
    y_i = 0
    for block_y_start in steps:
        block_y_end = block_y_start + step
        x_i = 0
        for block_x_start in steps:
            block_x_end = block_x_start + step
            block = image[int(block_y_start) : int(block_y_end), int(block_x_start): int(block_x_end)]
            pixel_color = np.average(block)
            new_image[y_i, x_i] = pixel_color
            x_i += 1
        y_i += 1
    return new_image

def encode_counting(image):
    result = np.zeros((0), dtype="uint8")
    length_of_color = 0
    is_white = image[0,0] != 0

    for y_i, row in enumerate(image):
        for x_i, pixel in enumerate(row):
            is_pixel_white = pixel != 0
            if (is_pixel_white == is_white):
                length_of_color += 1
            else:
                result = np.append(result, length_of_color)
                length_of_color = 1
                is_white = is_pixel_white
    result = np.append(result, length_of_color)

    return result


def decode_counting(encoded):
    result = np.zeros((0), dtype="uint8")
    is_white = False  # technical dept should be dynamic
    for section in encoded:
        result_section = np.full((section), int(is_white))
        is_white = not is_white
        result = np.concat((result, result_section))
    size = int(math.sqrt(len(result)))
    print(size)
    result = result[:size**2]
    result = result.reshape((size, size))
    return result

In [ ]:
# results
fig, ax = plt.subplots()

print("empty array size:", sys.getsizeof(np.array))

encoding_techique = ['base case',
                     'binairy w/ bits',
                     'binary w/ bits numpy',
                     "compression",
                     "counting changes"]
counts = [
    sys.getsizeof(train_images[20]),
    sys.getsizeof(binary_encode(train_images[20])),
    sys.getsizeof(binary_encode_np(train_images[20])),
    sys.getsizeof(compres_image(train_images[20])),
    sys.getsizeof(encode_counting(train_images[20]))]

ax.bar(encoding_techique, counts)

ax.set_ylabel('bytes')
ax.set_title('preformance getsizeof')
fig.set_size_inches((12, 5))



fig, ax = plt.subplots()

encoding_techique = ['base case',
                     'binairy w/ bits',
                     'binary w/ bits numpy',
                     "compression",
                     "counting changes"]
counts = [
    images[40].nbytes,
    sys.getsizeof(binary_encode(images[40])),
    binary_encode_np(images[40]).nbytes,
    compres_image(images[40]).nbytes,
    encode_counting(images[40]).nbytes]

ax.bar(encoding_techique, counts)

ax.set_ylabel('bytes')
ax.set_title('preformance np.nbytes')
fig.set_size_inches((12, 5))

plt.show()

fig, ax = plt.subplots()
encoding_techique = ['baseline',
                     'binairy w/ numpy',
                     "compression 24 px",
                     "compression 20 px",
                     "compression 14 px",
                     "compression + bin"]
counts = [
    88.7,
    88.3,
    86.2,
    85,
    87,
    83


    ]

ax.bar(encoding_techique, counts)

ax.set_ylabel('accuracy')
ax.set_title('accuracy for different compression methods')
fig.set_size_inches((12, 5))

plt.show()

Op basis van deze prestaties wordt de geconculdeerd dat binair binnen tot weinig dataverlies leid terwijl de memory footprint drastisch wordt gereduceerd. Daarom wordt deze vorm van compression ingezet. Daarmee kunnen meer afbeeldingen worden opgeslagen. Dit is extra voordelig bij algoritmen die baat hebben bij het opslaan van zo veel mogelijk afbeeldingen, zoals K-NN en K-means.

### Hoe veel abeeldingen passen er op de mystery device?

Voor k-nn en k-means is de groote van het model vrijwel geheel ingegeven door de groote van de afbeeldingen die moeten worden opgeslagen. Hoe meer afbeeldingen hoe naukeuriger het model wordt. Dus dan ontstaat de vraag, hoe veel afbeeldingen passen op de mystery device?

Op basis van eerder onderzoek in week 4 wordt de keuze gemaakt om gebruikt te maken van binning. Want deze zorgt voor een significante daling in het geheugengebruik met een minimale daling in de accuracy.\
Voor Er wordt verder gerekend met de zelf gemaakte binning om geheugengebruik te bepalen (omdat deze de overhead van de datasctuctuur meerekend) iets wat niet wordt gedaan bij numpy.\
Hierbij komen we uit op een geheugengebruik van 131 bytes.
128 kb bevat\
128_000 bytes\
Dat zijn dus 128_000 / 131 = __977.1 afbeeldingen__

Daarom is het voor de k-nn en k-means alogritmen logisch om voor 90 prototypes * 10 verschilledende digits te kiezen.

## memory footprint van python

De memory footprint van python is groot. Dit blijkt uit een onderzoek met behulp van een docker container. In dit korte hoofdstuk wordt uitgelegd hoe dit is bepaald.

Een docker container is gemaakt met behulp van `docker run --rm -p 8889:8888 -v "$(pwd):/home/jovyan/work" quay.io/jupyter/base-notebook start-notebook.py --NotebookApp.token='my-token'` In docker desktop kunnen vervolgens de gebruikte resources in kaart gebracht worden.\
De base load is 80 mb. Na het runnen van een eenvoudig script `print("Hello World")` stijgt de load naar 160 mb. Het runnen van de activatie functies opdracht binnen deze omgeving zorgt voor een constante load (ook na runnen) van 1.6 gb.

Op basis van deze bevindingen wordt geconcludeerd dat het onmogelijk is de gehele applicatie binnen de 128kb ram te kunnen runnen.

Daarom is besloten alleen de memory footprint voor de opslag van een model bij te houden. 

## Algoritme keuze
### Overzicht bestaande data

De verschillende toegepaste alogritmes zijn:

- Handgemaakte decision tree
- week 3 ding
- k-nearest neigbor
- k-means
- linear regression
- decision tree gemaakt met gini
- neuraal netwerkt met keras
- neuraal netwerkt handgemaakt

### Uitleg verschillende algoritmen
Hieronder wordt een korte uitleg gegeven van de verschillende algoritmen.
__Handgemaakte decision tree__: Een aantal agregaat features worden door een grote if/else if/elke keten gehaalt. Een latere aanpassing bleek beter resultaat te geven. Hierbij worden de outputs van features vergeleken met eerdere output van deze features. Vervolgens krijgt een nummer een punt als de feature het meest met het gemiddelde van dit getal overeen komt. Zo wordt door een lijst van features heen gegaan en kan uit eindelijk een meest waarschijnlijk getal worden gegeven.\
__data__: Model data opgeslagen als dict, input als 2d numpy array.

__week 3 ding__: Een zelf bedacht algoritme. Deze maakt een som van alle verschillen tussen de bekende afbeeldingen en de gegeven afbeelding. Vervolgens wordt het cijfer voorspeld die het kleinste verschil heeft.\
__data__: Model data opgeslagen als ordered numbers dict, input als 2d numpy array.

__k-nearest neigbor__: K-nearest neighbor zoekt de afbeeldingen die het meest lijken op de gegeven afbeelding. Vervolgens wordt het nummer voorspeld dat het vaakst terugkomt in deze nabij gelegen andere afbeeldingen (de buren).\
__data__:Model data opgeslagen als ordered numbers dict, input als 2d numpy array.

__k-means__: K means zoekt naar clusters in data. Kort door de bocht kan worden gezegd dat K-means de meest representative afbeeldingen maakt voor ieder cijfer. Vervolgens wordt het dichtst bijzijnde prototype van de gegeven afbeelding gezocht.\
__data__:Model data als ordered numbers dict, input als 2d numpy array.

__linear regression__: Linear regression is een vreemde keuze voor classificatieproblemen. Want linear regression zoekt een linear verband in een dataset (puntenwolk). In dit geval zoekt linear regression naar een verband tussen alle individuele pixelwaarden en of een afbeelding wel (1) of niet (0) een nummer representeerd. Vervolgens wordt de voorspelling met de meeste zekerheid gekozen.\
__data__: Model data opgeslagen in geimporteerde type, input als 2d numpy array.

__decision tree gemaakt met gini__: In tegenstelling tot de handgemaakte decision tree is deze machinaal gecontrueerd. Gini is een maatstaf voor de zuiverheid van data. Het doel bij het maken van de decision tree is dan het minimaliseren van de onzuiverheden in alle lagen (de dataset zo snel mogelijk zo zuiver mogelijk krijgen).\
__data__: Model data opgeslagen in geimporteerde type, input als 2d numpy array.

__neuraal netwerkt met keras__: Een neuraal netwerk maakt een beslissing op basis van wegingsfactoren en activation functions. Een input wordt door verschillende lagen van een neuraal netwerk heen geleid. In iedere laag worden inputs vermenigvuldigd met wegingsfactoren, ook wordt er een bias bij opgetelt. Daarna wordt een activator function ingezet, dit is een (vaak) eenvoudige functie (zoals RELU). Om het model goede voorspellingen te laten doen worden de wegingen in backpropegation aangepast om voorspelling te verbeteren.\
In dit geval is gebruik gemaakt van de library keras om het neuraal netwerk te bouwen.\
__data__: Model data opgeslagen in geimporteerde type, input als 2d numpy array.

__neuraal netwerkt handgemaakt__: Een neuraal netwerk maakt een beslissing op basis van wegingsfactoren en activation functions. Een input wordt door verschillende lagen van een neuraal netwerk heen geleid. In iedere laag worden inputs vermenigvuldigd met wegingsfactoren, ook wordt er een bias bij opgetelt. Daarna wordt een activator function ingezet, dit is een (vaak) eenvoudige functie (zoals RELU). Om het model goede voorspellingen te laten doen worden de wegingen in backpropegation aangepast om voorspelling te verbeteren.\
In dit geval is gebruik gemaakt van een handgemaakt neuraal netwerk.\
__data__: __NOG NADER TE BEPALEN__.



### Prestaties van de algoritmen
Eerst wordt onderzocht wat de hoogste accuracy is die ooit is gehaalt met deze techniek in het project. Daarna wordt onderzocht of de huidige implementatie op de mystery device past. Tot slot wordt de beste prestatie gegeven die binnen de perken van de mystery device past.\
Zoals besproken in "Encoding/memory footprint van python" wordt alleen gekeken naar de memory footprint van het model. Hoe veel geheugen wordt gebruikt tijdens het runnen wordt buiten beschouwing gelaten. Evenals overhead van NumPy en Python.
De resultaten van tabel 1 tonen al bekende gegevens op basis van eerder opdrachten.

__tabel 1__ vergelijking algoritmen
algoritme | Handgemaakte d tree |week 3 ding| k-nearest neighbor | k-means | linear regression | gini d tree | nn keras | nn handgemaakt|
----------|---------------------|-----------|--------------------|---------|-------------------|-------------|----------|---------------|
accuracy top| 27.8%             |72%        | onbekend           | 93%     | 86.0%             | 88.0%       | 97%      | onbekend      |
past op mystery device| ja      |nee        | onbekend           | afhankelijk| onbekend       | onbekend    | ja       | onbekend      |
beste passende prestaties| 27.8%|onbekend   | onbekend           | 88.3    | onbekend          | onbekend    | 79.5%    | onbekend      |

### Vervolg onderzoek

Zoals te zien is in tabel 1 zijn een aantal opties niet voledig afgewogen. In dit onderdeel worden deze onzekerheden weggewerkt zodat een voledige tabel ontstaat.

#### week 3 ding

In de opdracht van week 3 is een implementatie ontwikkeld die wat wegheeft van k-nearest neigbor. er wordt een accuracy van 57% gehaalt op op de test dataset.

In [ ]:
def calculate_min_differances(image):
    likeness  = {0: 999999999, 1: 999999999, 2: 999999999, 3: 999999999, 4: 999999999, 5: 999999999, 6: 999999999, 7: 999999999, 8: 999999999, 9: 999999999}
    for number in ordered_images:
        comparers = ordered_images[number]
        for compareable in comparers:
            differance = np.sum(np.absolute(image - compareable))
            if likeness[number] > differance:
                likeness[number] = differance
    return likeness

In [ ]:
count = 0
mistakes = 0;
for i, _ in enumerate(test_images_binned):
    count += 1
    if count > 5000:
        break;
    differances = calculate_min_differances(test_images_binned[i])
    guess = min(differances, key=differances.get)
    if (guess != test_labels[i]):
        mistakes += 1
print("Finished!")
print(f"mistakes: {mistakes}")
print(f"accuracy {mistakes / count * 100}")



Finished!
mistakes: 2858
accuracy 57.14857028594281


#### K-NN

In [66]:
# K-NN
def knn(image, K=5):
    likeness = {}
    for number in ordered_images:
        comparers = ordered_images[number]
        for compareable in comparers:
            likeness[number] = np.sum(np.absolute(image - compareable))
    likeness = dict(sorted(likeness.items()))
    # print(likeness)
    most_occurance = {0: 0, 1: 0, 2: 0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0}
    for i in range(K):
        most_occurance[list(likeness.keys())[i]] += 1
    return most_occurance

In [67]:
count = 0
mistakes = 0
for i, _ in enumerate(test_images_binned):
    count += 1
    if count > 5000:
        break;
    differances = knn(test_images_binned[i])

    guess = max(differances, key=differances.get)
    if (guess != test_labels[i]):
        mistakes += 1
print("Finished!")
print(f"mistakes: {mistakes}")
print(f"accuracy {mistakes / count * 100}")

Finished!
mistakes: 4540
accuracy 90.78184363127374


#### K-means

In [34]:
import math;
from random import choices;
import numpy as np;
import matplotlib.pyplot as plt;


def order_by_number(dataset_images, dataset_labels):
    """
    Seperate the mnist dataset into a dictionary ordered by label
    """
    ordered_images = {0: [], 1: [], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[], 9:[]}
    for i, _ in enumerate(dataset_images):
        ordered_images[dataset_labels[i]].append(dataset_images[i])
    return ordered_images


def dist(a: np.array, b: np.array):
    """
    calculate the distance between two points in N-dimensional space
    """
    abs_differances = np.abs(a - b)
    dist = math.sqrt(np.sum(abs_differances**2))
    return dist


def k_means(datapoints: tuple[np.array], clusters_amount: int) -> set[tuple]:
    """
    Find the cluster centers
    """
    clusterOrigins: np.array[np.array] = np.array(choices(datapoints, k=clusters_amount), dtype=np.half)
    new_origins = list()
    while True:

        clusters = dict()

        # reassign points to cluster_origins
        for point in datapoints:
            closest_cluster = min(clusterOrigins, key=lambda c : dist(c, point))

            closest_cluster = closest_cluster.tobytes()
            if (closest_cluster not in clusters.keys()):
                clusters[closest_cluster] = list()

            clusters[closest_cluster].append(point)

        # recalcuate origins
        new_origins.clear()
        for key in clusters:
            mean = np.mean(clusters[key], axis=0)
            new_origins.append(mean)

        # base case
        if np.array_equal(np.array(new_origins, dtype=np.half), clusterOrigins):
            return new_origins
        clusterOrigins = new_origins


def vector_to_image(vector: np.array):
    """
    Turn a vector into an image
    """
    length = len(vector)
    width = int(math.sqrt(length))
    square_vector = vector[:width**2]
    image = np.reshape(square_vector, (width, width))
    return image


def show_image(image):
    """
    Show a greyscale square image of any resolution. Image is stored in 2d array.
    """
    plt.imshow(image, cmap="grey")
    plt.show()

def make_prototype(train_images, train_labels, prototypes_per_number: int):
    ordered_images = order_by_number(train_images, train_labels)
    prototype: dict[int: np.array] = dict()
    for num in ordered_images:
        prototype[num] = k_means(ordered_images[num], prototypes_per_number)
        print(f"finished for number {num}")
    print("finished computing prototypes")
    return prototype

def get_distance(a: np.array, b: np.array):
    return dist(a, b)

def predict(image, prototypes):
    img_vector = normalize_and_vectorize_image(image)
    likeness  = {0: 999999999, 1: 999999999, 2: 999999999, 3: 999999999, 4: 999999999, 5: 999999999, 6: 999999999, 7: 999999999, 8: 999999999, 9: 999999999}
    for number in prototypes:
        prototypes_values = prototypes[number]
        for prototype in prototypes_values:
            differance = dist(prototype, img_vector)
            if likeness[number] > differance:
                likeness[number] = differance
    return min(likeness, key=lambda k: likeness[k])

def determine_accuracy(prototypes, images, labels):
    total = len(images)
    misses = 0
    miss_for_num: dict  = {0: 0, 1: 0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0}
    for i, image in enumerate(images):
        prediction = predict(image, prototypes)
        # print(f"pred: {prediction} | act: {labels[i]}")
        if prediction != labels[i]:
            misses += 1
            miss_for_num[labels[i]] += 1
    print(f"misses {misses}")
    print(f"accuracy {(total - misses) / total * 100} %")
    print(miss_for_num)



In [ ]:
prototypes = make_prototype(train_images_binned, train_labels, 25)

In [ ]:
determine_accuracy(prototypes, test_images_binned, test_labels)

#### Linear Regression

In [ ]:
# code gekopieërd uit P8
from sklearn.linear_model import LinearRegression


def normalize_image(image: np.array):
    """
    normalize values between 0 and 1
    """

    maximum = np.max(image)
    normalised: np.array = image / maximum

    return normalised


def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum
    return normalised


def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    print(f"coef {model.coef_.shape}")
    return model


def train_linear_model(X_train, y_binary):
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model

def determine_accuracy(prototypes, images, labels):
    total = len(images)
    misses = 0
    miss_for_num: dict  = {0: 0, 1: 0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0, 8:0, 9:0}
    for i, image in enumerate(images):
        prediction = predict(prototypes, image)
        # print(f"pred: {prediction} | act: {labels[i]}")
        if prediction != labels[i]:
            misses += 1
            miss_for_num[labels[i]] += 1
    print(f"misses {misses}")
    print(f"accuracy {(total - misses) / total * 100} %")
    print(miss_for_num)


def predict(models, image):
    results = {}
    for i in models:
        results[i] = models[i].predict([image])
    best_fit = max(results, key= lambda k: results[k])
    return best_fit



from sklearn.linear_model import LinearRegression

def train_linear_model(X_train, y_binary) -> LinearRegression:
    model = LinearRegression()
    model.fit(X_train, y_binary)
    return model



def train_numbers(train_images, train_labels):
    models = {}
    for num in range(10):
        models[num] = train_linear_model(train_images, [label == num for label in train_labels])
        print(f"finished for {num}")
    return models

models = train_numbers(train_images_vectored, train_labels)

determine_accuracy(models, test_images_vectored, test_labels)

finished
finished for 0
finished for 1
finished for 2
finished for 3
finished for 4
finished for 5
finished for 6
finished for 7
finished for 8
finished for 9
misses 1398
accuracy 86.02 %
{0: 36, 1: 28, 2: 219, 3: 131, 4: 101, 5: 233, 6: 83, 7: 144, 8: 215, 9: 208}


In [17]:
from pickle import dumps

def kb_size(obj):
    return sys.getsizeof(dumps(obj)) / 1000

print(kb_size(models))

127.195


Het blijkt dat het model met 127.195 kb net binnen het ramgeheugen past. Hiermee wordt aan de opdracht interpretatie voldaan dat alleen het model op het ram geheugen moet passen.


#### gini d tree

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.datasets import mnist;

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images_flattend = train_images.reshape((60_000, 784))
test_images_flatted = test_images.reshape((10_000, 784))

dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=20,
    random_state=42
)

dt.fit(train_images_flattend, train_labels)

y_pred = dt.predict(test_images_flatted)


acc = accuracy_score(y_pred, test_labels)
print("Decision Tree Accuracy on MNIST:", acc)

print("\nClassification Report:")
print(classification_report(y_pred, test_labels))

Decision Tree Accuracy on MNIST: 0.8818

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.91      0.93      1008
           1       0.96      0.96      0.96      1140
           2       0.86      0.87      0.87      1014
           3       0.85      0.84      0.84      1031
           4       0.88      0.88      0.88       989
           5       0.84      0.84      0.84       893
           6       0.89      0.90      0.89       945
           7       0.91      0.91      0.91      1022
           8       0.81      0.83      0.82       955
           9       0.86      0.86      0.86      1003

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [18]:
print(kb_size(dt))

1034.074


Dit model is te groot. We maken het kleiner. Hieronder wordt de grens van 128 kb zo dicht mogelijk benaderd.

In [29]:
dt_v2 = DecisionTreeClassifier(
    criterion='gini',
    max_depth=8,
    random_state=42
)

dt_v2.fit(train_images_flattend, train_labels)

y_pred = dt_v2.predict(test_images_flatted)


acc = accuracy_score(y_pred, test_labels)
print("Decision Tree Accuracy on MNIST:", acc)

print("\nClassification Report:")
print(classification_report(y_pred, test_labels))

Decision Tree Accuracy on MNIST: 0.8185

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      1019
           1       0.95      0.87      0.90      1242
           2       0.80      0.86      0.83       954
           3       0.76      0.81      0.79       948
           4       0.79      0.84      0.81       924
           5       0.76      0.70      0.73       981
           6       0.84      0.80      0.82      1007
           7       0.81      0.90      0.85       931
           8       0.70      0.80      0.74       852
           9       0.82      0.73      0.77      1142

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



In [30]:
print(kb_size(dt_v2))

73.873


Het model past goed binnen de 128 KB. Maar de accuracy is nog maar 88.2%

#### nn handgemaakt



__tabel 2__ vergelijking algoritmen met nieuwe data
algoritme | Handgemaakte d tree | week 3 ding     |k-NN            | k-means | linear regression | gini d tree | nn keras | nn handgemaakt|
----------|---------------------|-----------------|----------------|---------|-------------------|-------------|----------|---------------|
accuracy top| 27.8%             | 72%             |onbekend        | 93%     | 86.0%             | 88.0%       | 97%      | onbekend      |
past op mystery device| ja      | met aanpassingen|met aanpassingen| ja      | ja                | ja          | ja       | onbekend      |
beste passende prestaties| 27.8%| 61%             |91%             | 88.3%   | 86.05%            | 81.2%       | 79.5%    | onbekend      |

### XXXXX is het beste algoritme